In [1]:
import pandas as pd
import numpy as np
import scipy

In [15]:
np.random.seed(42)

In [16]:
# session_id | user_id | date       | group     | clicks | time_on_page | booked | revenue
# -----------|---------|------------|-----------|--------|--------------|--------|--------
# 1          | 101     | 2026-03-01 | control   | 3      | 120          | 0      | 0
# 2          | 102     | 2026-03-01 | treatment | 7      | 240          | 1      | 320.5
# ...

In [38]:
session_id = np.arange(1,1001)
dates = pd.date_range("2026-03-01","2026-03-31")
size = len(session_id)
user_id_control = np.arange(101,201)
user_id_treatment = np.arange(201,301)

print(len(user_id_control),len(user_id_treatment))

dates_lst = np.random.choice(dates,size)
user_control_lst = np.random.choice(user_id_control,int(size/2))
user_treatment_lst = np.random.choice(user_id_treatment,int(size/2))

total_user_id = np.concatenate((user_control_lst, user_treatment_lst))

# print(total_user_id)
# print(len(user_control_lst))
# print(len(total_user_id))
# print(dates)

fake_df = pd.DataFrame({"session_id":session_id,
              "user_id":total_user_id,
              "date":dates_lst,})

fake_df['group'] = np.where(fake_df['user_id'].isin(user_id_control),"control","treatment")

fake_df['booked'] = np.where(fake_df['group']=='treatment',
                             np.random.binomial(1,0.3,size),
                             np.random.binomial(1,0.2,size)) 


fake_df['time_on_page'] = np.where(fake_df['group']=='treatment',
                             np.random.randint(1,300,size=size),
                             np.random.randint(50,400,size)) + np.random.randint(25,100,size)

fake_df['clicks'] = (fake_df['time_on_page']/10 + np.random.randint(1,5,size)).astype('int')

fake_df['revenue'] = np.where(fake_df['booked']==1,(200 + np.random.randint(50,150,size) - fake_df['clicks']*0.5), 0) 

print(fake_df)

100 100
     session_id  user_id       date      group  booked  time_on_page  clicks  \
0             1      117 2026-03-15    control       0           322      34   
1             2      134 2026-03-06    control       1           339      34   
2             3      151 2026-03-26    control       0           148      17   
3             4      167 2026-03-05    control       1           477      51   
4             5      124 2026-03-29    control       1           271      30   
..          ...      ...        ...        ...     ...           ...     ...   
995         996      274 2026-03-16  treatment       0           318      35   
996         997      287 2026-03-02  treatment       0           164      20   
997         998      233 2026-03-21  treatment       0           330      36   
998         999      243 2026-03-30  treatment       0           274      29   
999        1000      283 2026-03-31  treatment       0           180      19   

     revenue  
0        0.0  
1

In [43]:
fake_df.groupby("group").agg(
    count_sessions = ("session_id","count"),
    avg_revenue = ("revenue","mean"),
    avg_clicks = ("clicks","mean"),
    avg_time = ("time_on_page","mean"),
    avg_booking =("booked","mean")
                             )

,count_sessions,avg_revenue,avg_clicks,avg_time,avg_booking
group,,,,,
control,500,59.709,31.320,292.536,0.214
treatment,500,91.046,23.314,211.906,0.314


In [65]:
# primary metric 
# avg revenue? revenue/clicks? booked?
# 2 guardrails: clicks & avg time 

from scipy import stats
treatment_s = fake_df[fake_df['group']=='treatment']['booked']
control_s = fake_df[fake_df['group']=='control']['booked']

t_stat, p_value = stats.ttest_ind(control_s,treatment_s,equal_var=False,alternative='greater')

print(t_stat, p_value)

-3.6066703119191907 0.9998371248359763


In [66]:
from statsmodels.stats.power import TTestIndPower

effect_size = (treatment_s.mean() - control_s.mean()) / control_s.std()

analysis= TTestIndPower()

power = analysis.solve_power(effect_size=effect_size,
              nobs1=len(treatment_s),
              alpha=0.05,
              alternative="larger")
# print(power)

print(f"Effect size: {effect_size:.4f}")
print(f"Achieved power: {power:.4f}")

Effect size: 0.2436
Achieved power: 0.9862


In [68]:
required_n = analysis.solve_power(
    effect_size=effect_size,
    power=0.8,
    alpha=0.05,
    alternative='larger'
)

print(f"Required users per group: {required_n:.0f}")

Required users per group: 209


In [69]:
treatment_s = fake_df[fake_df['group']=='treatment']['revenue']
control_s = fake_df[fake_df['group']=='control']['revenue']

t_stat, p_value = stats.ttest_ind(control_s,treatment_s,equal_var=False,alternative='greater')

print(t_stat, p_value)

-3.9370982416467877 0.9999558235172892


In [70]:
effect_size = (treatment_s.mean() - control_s.mean()) / control_s.std()

analysis= TTestIndPower()

power = analysis.solve_power(effect_size=effect_size,
              nobs1=len(treatment_s),
              alpha=0.05,
              alternative="larger")
# print(power)

print(f"Effect size: {effect_size:.4f}")
print(f"Achieved power: {power:.4f}")

Effect size: 0.2721
Achieved power: 0.9960
